In [1]:
import json

import requests

# --- Config: change these for any NSE-listed symbol ---
SYMBOL = "OMKARCHEM"
FROM_DATE = "01-06-2026"
TO_DATE = "30-06-2026"

_API_URL = "https://www.nseindia.com/api/NextApi/apiClient/GetQuoteApi"
_HOMEPAGE = "https://www.nseindia.com"
_PAGE_URL = "https://www.nseindia.com/companies-listing/corporate-filings-announcements"

# NSE blocks generic Python user agents without a session cookie.
_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": _PAGE_URL,
}

params = {
    "functionName": "getCorporateAnnouncement",
    "symbol": SYMBOL,
    "marketApiType": "equities",
    "subject": "",
    "fromDate": FROM_DATE,
    "toDate": TO_DATE,
}

session = requests.Session()
session.headers.update(_HEADERS)

# Warm up cookies from the homepage before hitting the API.
session.get(_HOMEPAGE, timeout=30)

response = session.get(_API_URL, params=params, timeout=30)
response.raise_for_status()

result = response.json()

by_desc: dict[str, list[dict]] = {}
for item in result:
    desc = (item.get("desc") or "(missing)").strip()
    by_desc.setdefault(desc, []).append(item)

print(f"{len(result)} announcements across {len(by_desc)} desc buckets\n")
for desc, items in sorted(by_desc.items(), key=lambda kv: (-len(kv[1]), kv[0])):
    print(f"{desc}: {len(items)}")

print("\n" + "=" * 60 + "\n")
print(json.dumps(by_desc, indent=2))

3 announcements across 2 desc buckets

Copy of Newspaper Publication: 2
Outcome of Board Meeting: 1


{
  "Copy of Newspaper Publication": [
    {
      "symbol": "OMKARCHEM",
      "desc": "Copy of Newspaper Publication",
      "dt": "14062026165506",
      "attchmntFile": "https://nsearchives.nseindia.com/corporate/OMKARCHEM_14062026165448_Intimation_Newspaper_Publication.pdf",
      "sm_name": "Omkar Speciality Chemicals Limited",
      "sm_isin": "INE474L01016",
      "an_dt": "14-Jun-2026 16:55:06",
      "sort_date": "2026-06-14 16:55:06",
      "seq_id": null,
      "smIndustry": "Chemicals - Speciality",
      "orgid": null,
      "attchmntText": "Omkar Speciality Chemicals Limited has informed the Exchange about Copy of Newspaper Publication for Audited Standalone Financial Results for the Quarter & Financial Year ended March 31, 2026",
      "bflag": null,
      "old_new": null,
      "csvName": null,
      "exchdisstime": "14-Jun-2026 16:55:07",
      "difference": "00:00:01

In [2]:
from collections import defaultdict


def classify_announcement(item: dict) -> str:
    """Recategorize using attchmntText (with desc/url as secondary hints)."""
    text = (item.get("attchmntText") or "").strip()
    desc = (item.get("desc") or "").strip()
    t = text.lower()

    def has(*phrases: str) -> bool:
        return any(p in t for p in phrases)

    if t.strip().strip("'\".") == "statement":
        return "Generic Statement"

    # Earnings call
    if has("transcript of the discussion", "earnings call transcript", "concall transcript"):
        return "Earnings Call Transcript"
    if has("audio recording of the discussion", "audio / video recording and transcript"):
        return "Earnings Call Audio Recording"
    if has("will hold", "will be held", "scheduled to be held") and has(
        "conference call", "earnings conference call", "earnings call", "concall"
    ):
        return "Earnings Call Intimation"
    if has("scheduled to be held") and has("board of directors", "meeting of the board"):
        return "Board Meeting Intimation"

    # Compliance / ancillary filings (not the results PDF itself)
    if desc == "Monitoring Agency Report" or has("monitoring agency report", "monitoring agency"):
        return "Monitoring Agency Report"

    # AGM / shareholder meetings
    if has("voting results", "scrutiniser", "scrutinizer", "regulation 44(3)"):
        return "AGM Voting Results"
    if has("gist of the proceedings") and has("general meeting"):
        return "AGM Proceedings"
    if has("postal ballot notice"):
        return "Postal Ballot Notice"
    if has("notice of") and has("annual general meeting"):
        return "AGM Notice"
    if has("annual general meeting") and has("will be held", "wilI be held", "held on"):
        return "AGM Intimation"
    if has("chairman's statement", "chairman s statement") and has("general meeting"):
        return "AGM Chairman Statement"
    if has("presentation on item") and has("annual general meeting", "agm", "general meeting"):
        return "AGM Presentation"

    # Corporate actions
    if desc == "Dividend" or (has("dividend") and has("board of directors") and not has("financial results")):
        return "Dividend"
    if desc == "Record Date" or has("record date for the purpose of"):
        return "Record Date"

    # Financial results (actual filing, not a call intimation or prior notice)
    if has(
        "financial results",
        "submitted to the exchange, the consolidated",
        "submitted to the exchange, the financial results",
        "unaudited financial results",
        "audited financial results",
        "integrated filing (financial)",
    ) and not has("scheduled to be held", "will hold", "will be held", "conference call"):
        return "Financial Results"
    if desc in ("Outcome of Board Meeting", "Press Release") and has("financial results"):
        if not has("will hold", "will be held", "conference call", "prior intimation"):
            return "Financial Results"

    # Investor relations
    if has(
        "institutional investors' meeting",
        "institutional investors meeting",
        "executives participated in the institutional",
        "will be participating in the institutional",
        "schedule of meet",
    ):
        return "Investor Meet"
    if desc == "Investor Presentation" or has("investor presentation"):
        return "Investor Presentation"

    # Press / media
    if has("media release", "media statement") or (desc == "Press Release" and not has("financial results")):
        return "Press / Media Release"

    # M&A / restructuring
    if has("initial public offer", "proposed ipo"):
        return "IPO / Listing"
    if desc == "Credit Rating" or has("credit rating"):
        return "Credit Rating"
    if desc == "Trading Window" or has("trading window"):
        return "Trading Window"
    if desc == "ESOP/ESOS/ESPS" or (has("allotment of") and has("shares")):
        return "ESOP / Share Allotment"
    if desc == "Appointment" or has("appointment of"):
        return "Director Appointment"
    if desc == "Change in Director(s)" or has("change in director"):
        return "Director Change"
    if desc == "Acquisition" or has("acquisition of", "update on acquisition"):
        return "Acquisition"
    if desc in ("Sale or disposal", "Diversification/Disinvestment") or has(
        "sale of",
        "voluntary liquidation",
        "voluntary winding up",
        "ceasing to be",
    ):
        return "Sale / Disposal"
    if desc == "Incorporation" or has("incorporation of"):
        return "Subsidiary Incorporation"
    if has("national company law tribunal", " nclt "):
        return "Court / Tribunal Order"
    if desc == "Other Restructuring" or has("amalgamation", "merger of", "restructuring", "step-down subsidiary"):
        return "Corporate Restructuring"

    # Compliance / admin
    if desc == "Copy of Newspaper Publication" or has("newspaper clipping"):
        return "Newspaper Publication"
    if desc == "News Verification" or has("sought clarification"):
        return "News Verification"
    if desc == "Action(s) taken or orders passed":
        return "Regulatory Action / Orders"
    if has("business responsibility and sustainability report", "brsr"):
        return "Sustainability Report (BRSR)"
    if has("disclosure under regulation 30", "regulation 30"):
        return "Reg 30 Disclosure"

    return desc or "Other"


recategorized: dict[str, list[dict]] = defaultdict(list)
for item in result:
    bucket = classify_announcement(item)
    enriched = {**item, "nse_desc": item.get("desc"), "event_bucket": bucket}
    recategorized[bucket].append(enriched)

print(f"{len(result)} announcements -> {len(recategorized)} event buckets\n")
for bucket, items in sorted(recategorized.items(), key=lambda kv: (-len(kv[1]), kv[0])):
    print(f"{bucket}: {len(items)}")

moved = [
    item
    for bucket_items in recategorized.values()
    for item in bucket_items
    if item["nse_desc"] != item["event_bucket"]
]
print(f"\n{len(moved)} announcements recategorized away from NSE desc\n")

print("=" * 60)
print(json.dumps(recategorized, indent=2))

3 announcements -> 2 event buckets

Financial Results: 2
Newspaper Publication: 1

3 announcements recategorized away from NSE desc

{
  "Financial Results": [
    {
      "symbol": "OMKARCHEM",
      "desc": "Copy of Newspaper Publication",
      "dt": "14062026165506",
      "attchmntFile": "https://nsearchives.nseindia.com/corporate/OMKARCHEM_14062026165448_Intimation_Newspaper_Publication.pdf",
      "sm_name": "Omkar Speciality Chemicals Limited",
      "sm_isin": "INE474L01016",
      "an_dt": "14-Jun-2026 16:55:06",
      "sort_date": "2026-06-14 16:55:06",
      "seq_id": null,
      "smIndustry": "Chemicals - Speciality",
      "orgid": null,
      "attchmntText": "Omkar Speciality Chemicals Limited has informed the Exchange about Copy of Newspaper Publication for Audited Standalone Financial Results for the Quarter & Financial Year ended March 31, 2026",
      "bflag": null,
      "old_new": null,
      "csvName": null,
      "exchdisstime": "14-Jun-2026 16:55:07",
      "dif

In [3]:
import hashlib
import io
import re
from urllib.parse import urlparse

_PDF_MAGIC = b"%PDF-"
_NUMBER_RE = re.compile(r"(?<![\d.])(-?\d[\d,]*(?:\.\d+)?)(?![\d.])")

# url -> (pdf_hash, classification dict, pdf_bytes)
_PDF_CLASSIFICATION_CACHE: dict[str, tuple[str, dict, bytes]] = {}


def parse_file_size_kb(raw: str | None) -> float | None:
    """Parse NSE fileSize strings like '32.83 MB' or '220.26 KB'."""
    if not raw:
        return None
    m = re.match(r"^\s*([\d.]+)\s*(KB|MB|GB)\s*$", raw.strip(), re.IGNORECASE)
    if not m:
        return None
    value = float(m.group(1))
    unit = m.group(2).upper()
    if unit == "KB":
        return value
    if unit == "MB":
        return value * 1024
    return value * 1024 * 1024


def build_text_blob(item: dict) -> str:
    """Lowercased concatenation of announcement text fields + URL basename."""
    url = (item.get("attchmntFile") or "").strip()
    parts: list[str] = []
    for key in ("attchmntText", "desc", "smIndustry", "subject"):
        value = item.get(key)
        if isinstance(value, str) and value:
            parts.append(value)
    if url:
        parts.append(url.rsplit("/", 1)[-1])
    return " ".join(parts).lower()


def is_pdf_url(url: str) -> bool:
    path = url.split("?", 1)[0].lower()
    return path.endswith(".pdf")


def download_pdf(url: str, session: requests.Session, *, timeout: int = 120) -> bytes:
    """Download a PDF from NSE archives using the warmed session."""
    response = session.get(url, timeout=timeout)
    response.raise_for_status()
    data = response.content
    if not data:
        raise ValueError(f"Empty response from {url}")
    if not data.startswith(_PDF_MAGIC):
        raise ValueError(f"Not a PDF (no %PDF- header): {url}")
    return data


def pdf_hash(pdf_bytes: bytes) -> str:
    return hashlib.sha256(pdf_bytes).hexdigest()


In [4]:
from pypdf import PdfReader

_POSITIVE_PHRASES = (
    r"statement of profit and loss",
    r"statement of (?:consolidated|standalone|unaudited|audited)?\s*financial",
    r"standalone statement of profit and loss",
    r"balance sheet",
    r"cash flow statement",
    r"(?:standalone|consolidated)\s+(?:unaudited|audited)\s+financial results",
    r"financial\s+r[eé]?sults for the (?:quarter|period|year) ended",
    r"financial results for the quarter and financial year",
    r"\bind as\b",
    r"regulation 33",
    r"limited review report",
    r"\bin crorel?\b",
    r"₹\s*in\s*(?:crore|lakh|million)",
    r"profit before tax",
    r"profit after tax",
    r"revenue from operations",
    r"independent auditor's report",
)

# Hard reject: ancillary compliance docs that mention finance but are not results packs.
_EXCLUDED_CONTENT_PHRASES = (
    r"monitoring agency report",
    r"monitoring agency",
    r"utilization of proceeds",
    r"certificate under regulation 74",
    r"regulation 74\s*\(\s*5\s*\)",
    r"credit rating",
    r"postal ballot",
)

_CORE_FINANCIAL_PHRASES = (
    r"statement of profit and loss",
    r"statement of (?:consolidated|standalone|unaudited|audited)?\s*financial",
    r"financial\s+r[eé]?sults for the (?:quarter|period|year) ended",
    r"(?:standalone|consolidated)\s+(?:unaudited|audited)\s+financial results",
    r"revenue from operations",
    r"profit before tax",
    r"profit after tax",
)

_NEGATIVE_PHRASES = (
    r"record date for the purpose of",
    r"annual general meeting.{0,40}will be held",
    r"appointment of (?:mr\.|ms\.|shri)",
    r"change in director",
    r"conference call.{0,60}intimation",
    r"conference call will be held",
    r"intimation of conference call",
    r"link of recording",
    r"re-appointment of",
    r"trading window",
    r"prior intimation",
    r"scheduled to be held",
    r"dear sir/madam",
    r"kindly take the same on record",
    r"dial in and other details",
    r"monitoring agency",
    r"utilization of proceeds",
)

_POSITIVE_RES = [re.compile(p, re.IGNORECASE) for p in _POSITIVE_PHRASES]
_NEGATIVE_RES = [re.compile(p, re.IGNORECASE | re.DOTALL) for p in _NEGATIVE_PHRASES]
_EXCLUDED_CONTENT_RES = [re.compile(p, re.IGNORECASE) for p in _EXCLUDED_CONTENT_PHRASES]
_CORE_FINANCIAL_RES = [re.compile(p, re.IGNORECASE) for p in _CORE_FINANCIAL_PHRASES]


def _is_financial_table_row(line: str) -> bool:
    numbers: list[float] = []
    for m in _NUMBER_RE.finditer(line):
        raw = m.group(1).replace(",", "")
        try:
            val = float(raw)
        except ValueError:
            continue
        if 1900 <= val <= 2099 and val == int(val):
            continue
        numbers.append(val)
    material = [n for n in numbers if abs(n) >= 500]
    return len(material) >= 2


def _extract_pdf_text(pdf_bytes: bytes, *, max_pages: int = 20) -> tuple[str, int]:
    reader = PdfReader(io.BytesIO(pdf_bytes))
    page_count = len(reader.pages)
    # Sample early pages (cover/auditor) plus a window where P&L tables usually start.
    sample_indices: list[int] = []
    for i in range(min(max_pages, page_count)):
        if i not in sample_indices:
            sample_indices.append(i)
    for i in range(6, min(page_count, 22)):
        if i not in sample_indices:
            sample_indices.append(i)
    sample_indices.sort()

    chunks: list[str] = []
    for i in sample_indices:
        try:
            chunks.append(reader.pages[i].extract_text() or "")
        except Exception:
            chunks.append("")
    return "\n".join(chunks), page_count


def classify_pdf_content(pdf_bytes: bytes, *, max_pages: int = 20) -> dict:
    """Heuristic classifier: is this PDF a financial results filing?"""
    text, page_count = _extract_pdf_text(pdf_bytes, max_pages=max_pages)
    lower = text.lower()
    text_len = len(lower)

    positive_hits = sum(1 for pat in _POSITIVE_RES if pat.search(lower))
    negative_hits = sum(1 for pat in _NEGATIVE_RES if pat.search(lower))
    excluded_hits = sum(1 for pat in _EXCLUDED_CONTENT_RES if pat.search(lower))
    has_core_financial = any(pat.search(lower) for pat in _CORE_FINANCIAL_RES)
    table_row_count = sum(1 for line in lower.splitlines() if _is_financial_table_row(line))

    score = 0.0
    reasons: list[str] = []

    if excluded_hits:
        reasons.append(f"{excluded_hits} excluded doc phrase(s)")
        return {
            "is_financial_report": False,
            "confidence": 0.0,
            "document_kind": "EXCLUDED",
            "signals": {
                "positive_hits": positive_hits,
                "negative_hits": negative_hits,
                "excluded_hits": excluded_hits,
                "has_core_financial": has_core_financial,
                "table_row_count": table_row_count,
                "page_count": page_count,
                "text_len": text_len,
            },
            "reasons": reasons,
        }

    score += min(positive_hits * 0.12, 0.48)
    if positive_hits:
        reasons.append(f"{positive_hits} financial phrase(s)")

    score -= min(negative_hits * 0.12, 0.36)
    if negative_hits:
        reasons.append(f"{negative_hits} letter/admin phrase(s)")

    # Integrated filings are long; outcome letters are 1-3 pages.
    if page_count >= 20:
        score += 0.42
        reasons.append(f"{page_count} pages (integrated filing)")
    elif page_count >= 10:
        score += 0.28
        reasons.append(f"{page_count} pages")
    elif page_count >= 5:
        score += 0.10
    elif page_count <= 3:
        score -= 0.25
        reasons.append(f"short doc ({page_count} pages)")

    if table_row_count >= 8:
        score += 0.28
        reasons.append(f"{table_row_count} table-like rows")
    elif table_row_count >= 3:
        score += 0.14
    elif table_row_count < 2 and page_count <= 5:
        score -= 0.20
        reasons.append("few numeric table rows")

    # Cover pages are often image-only; reward dense text on long docs.
    if text_len > 12000:
        score += 0.10
    elif text_len < 1200 and page_count <= 4:
        score -= 0.15
        reasons.append("minimal extractable text")

    # Short letters may mention "financial results" in the body — require tables or length.
    if page_count <= 4 and table_row_count < 4:
        score -= 0.18
        reasons.append("letter-shaped (short + no tables)")

    confidence = max(0.0, min(1.0, score))
    is_financial_report = confidence >= 0.55 and has_core_financial
    if confidence >= 0.55 and not has_core_financial:
        reasons.append("missing core financial-results sections")

    if is_financial_report:
        document_kind = "FINANCIAL_RESULT"
    elif negative_hits >= 2 or (page_count <= 3 and table_row_count < 2):
        document_kind = "LETTER"
    else:
        document_kind = "OTHER"

    return {
        "is_financial_report": is_financial_report,
        "confidence": round(confidence, 3),
        "document_kind": document_kind,
        "signals": {
            "positive_hits": positive_hits,
            "negative_hits": negative_hits,
            "excluded_hits": excluded_hits,
            "has_core_financial": has_core_financial,
            "table_row_count": table_row_count,
            "page_count": page_count,
            "text_len": text_len,
        },
        "reasons": reasons,
    }


In [5]:
_URL_FIN_MARKERS = (
    "bsenseoutcome",
    "outcomesigned",
    "outcome",
    "sefr_",
    "se_result",
    "finresult",
    "mediarelease",
    "_mr.",
    "financialresult",
    "fin_result",
    "finresult",
)

_URL_EXCLUDED_MARKERS = (
    "monitoring_agency",
    "certificate745",
    "certificate_74",
    "reg74",
    "grantdisclosure",
    "priorintimation",
    "changeindirector",
    "investorpresentation",
    "transcript",
    "concall",
    "shlsigned",
    "shareholdersletter",
    "shareholderletter",
)

_EXCLUDED_EVENT_BUCKETS = frozenset({
    "Monitoring Agency Report",
    "AGM Intimation",
    "Record Date",
    "Director Appointment",
    "Director Change",
    "Board Meeting Intimation",
    "Earnings Call Intimation",
    "Earnings Call Audio Recording",
    "Earnings Call Transcript",
    "Investor Presentation",
    "Credit Rating",
    "Certificate under SEBI (Depositories and Participants) Regulations, 2018",
    "ESOP / Share Allotment",
})

_MISLINKED_BUCKETS = _EXCLUDED_EVENT_BUCKETS

_PERIOD_ENDED_RE = re.compile(
    r"(?:period|quarter|year|financial year)\s+ended\s+"
    r"(\d{1,2}\s+[a-z]+\s+\d{4}|[a-z]+\s+\d{1,2},?\s+\d{4})",
    re.IGNORECASE,
)
_FY_TOKEN_RE = re.compile(r"q([1-4])fy(\d{2,4})", re.IGNORECASE)
_DATE_IN_URL_RE = re.compile(r"(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*[_\s]?20\d{2}", re.IGNORECASE)


def infer_period_markers(announcements: list[dict]) -> list[str]:
    """Derive period text markers from the announcement batch (symbol-agnostic)."""
    markers: set[str] = set()
    for item in announcements:
        for field in ("attchmntText", "desc"):
            text = item.get(field) or ""
            for m in _PERIOD_ENDED_RE.finditer(text):
                raw = m.group(1).lower().replace(",", "").strip()
                markers.add(raw)
                # Also add dd month yyyy and month dd yyyy variants.
                parts = raw.split()
                if len(parts) == 3:
                    markers.add(f"{parts[1]} {parts[0]} {parts[2]}")
        url = (item.get("attchmntFile") or "").lower()
        for m in _FY_TOKEN_RE.finditer(url):
            markers.add(f"q{m.group(1)}fy{m.group(2)}")
        for m in _DATE_IN_URL_RE.finditer(url):
            markers.add(m.group(0).replace("_", " "))
    return sorted(markers)


def _is_excluded_candidate(item: dict) -> bool:
    bucket = item.get("event_bucket") or item.get("desc") or ""
    if bucket in _EXCLUDED_EVENT_BUCKETS:
        return True
    url_lower = (item.get("attchmntFile") or "").lower()
    if any(m in url_lower for m in _URL_EXCLUDED_MARKERS):
        return True
    text = build_text_blob(item)
    if "monitoring agency" in text or "certificate under regulation 74" in text:
        return True
    return False


def metadata_score(item: dict) -> float:
    text = build_text_blob(item)
    url_lower = (item.get("attchmntFile") or "").lower()
    score = 0.0

    if "submitted to the exchange, the financial results" in text:
        score += 5.0
    elif "submitted to the exchange, the consolidated" in text:
        score += 4.0
    elif "outcome of the board meeting" in text and "financial results" in text:
        score += 3.5
    elif "financial results" in text and "scheduled to be held" not in text:
        if "conference call" not in text and "will hold" not in text:
            score += 2.0

    size_kb = parse_file_size_kb(item.get("fileSize") or item.get("attFileSize"))
    if size_kb is not None:
        if size_kb > 2048:
            score += 3.0
        elif size_kb > 800:
            score += 1.0
        elif size_kb < 300:
            score -= 2.0

    if any(m in url_lower for m in _URL_FIN_MARKERS):
        score += 2.0
    if any(m in url_lower for m in _URL_EXCLUDED_MARKERS):
        score -= 4.0

    if item.get("event_bucket") == "Financial Results":
        score += 1.5
    if item.get("event_bucket") == "Outcome of Board Meeting" or item.get("desc") == "Outcome of Board Meeting":
        score += 2.5

    if item.get("event_bucket") in _MISLINKED_BUCKETS:
        score -= 3.0

    return score


def _matches_period_markers(item: dict, period_markers: list[str] | None) -> bool:
    if not period_markers:
        return True
    text = build_text_blob(item)
    return any(m.lower() in text for m in period_markers)


def build_candidate_pool(
    announcements: list[dict],
    *,
    period_markers: list[str] | None = None,
) -> list[dict]:
    def _collect(markers: list[str] | None) -> list[dict]:
        seen_urls: set[str] = set()
        pool: list[dict] = []
        for item in announcements:
            url = (item.get("attchmntFile") or "").strip()
            if not url or not is_pdf_url(url):
                continue
            if _is_excluded_candidate(item):
                continue
            if url in seen_urls:
                continue
            if not _matches_period_markers(item, markers):
                continue
            seen_urls.add(url)
            pool.append({**item, "_metadata_score": metadata_score(item)})
        pool.sort(key=lambda x: x["_metadata_score"], reverse=True)
        return pool

    pool = _collect(period_markers)
    # If period markers are too strict for this symbol, fall back to the full batch.
    if not pool and period_markers:
        pool = _collect(None)
    return pool


def classify_pdf_url(url: str, session: requests.Session) -> dict:
    """Download (or cache) and classify a PDF by URL."""
    if url in _PDF_CLASSIFICATION_CACHE:
        _, classification, _ = _PDF_CLASSIFICATION_CACHE[url]
        return classification

    pdf_bytes = download_pdf(url, session)
    digest = pdf_hash(pdf_bytes)

    for cached_url, (cached_hash, cached_cls, cached_bytes) in _PDF_CLASSIFICATION_CACHE.items():
        if cached_hash == digest:
            _PDF_CLASSIFICATION_CACHE[url] = (digest, cached_cls, cached_bytes)
            return cached_cls

    classification = classify_pdf_content(pdf_bytes)
    _PDF_CLASSIFICATION_CACHE[url] = (digest, classification, pdf_bytes)
    return classification


def resolve_financial_report_pdf(
    announcements: list[dict],
    *,
    period_markers: list[str] | None = None,
    max_candidates: int = 8,
    session: requests.Session,
    tried_urls: set[str] | None = None,
) -> dict | None:
    """Search announcement batch for the PDF that actually is a financial report."""
    tried = tried_urls or set()
    pool = build_candidate_pool(announcements, period_markers=period_markers)

    for candidate in pool[:max_candidates]:
        url = candidate["attchmntFile"]
        if url in tried:
            continue
        tried.add(url)
        try:
            if url in _PDF_CLASSIFICATION_CACHE:
                digest, classification, pdf_bytes = _PDF_CLASSIFICATION_CACHE[url]
            else:
                pdf_bytes = download_pdf(url, session)
                digest = pdf_hash(pdf_bytes)
                classification = classify_pdf_content(pdf_bytes)
                _PDF_CLASSIFICATION_CACHE[url] = (digest, classification, pdf_bytes)
        except Exception as exc:
            print(f"  skip {url.rsplit('/', 1)[-1]}: {exc}")
            continue

        if classification["is_financial_report"]:
            return {
                "url": url,
                "announcement": candidate,
                "classification": classification,
                "pdf_hash": digest,
                "pdf_bytes": pdf_bytes,
                "metadata_score": candidate["_metadata_score"],
            }

    return None


In [6]:
# --- Demo: classify PDFs + resolve canonical financial report (any symbol) ---

import importlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "v3"))
import store

importlib.reload(store)
from store import save_resolved_financial_pdf


def _print_processing(proc: dict) -> None:
    if proc.get("success"):
        print(
            f"  Processing: {proc.get('extracted_values', 0)} extracted_values, "
            f"{proc.get('metric_values', 0)} metric_values, {proc.get('signals', 0)} signals"
        )
    elif proc:
        print(f"  Processing failed: {proc.get('error')}")
    else:
        print("  Processing: not run (restart kernel if you just updated v3 code)")

all_enriched = [
    item
    for bucket_items in recategorized.values()
    for item in bucket_items
]

period_markers = infer_period_markers(all_enriched)
print(f"Symbol: {SYMBOL}  |  Period markers inferred: {period_markers}\n")

print("PDF classification summary:\n")
print(f"{'event_bucket':<35} {'basename':<40} {'size':>10} {'meta':>5} {'is_fr':>6} {'conf':>5}")
print("-" * 105)

for item in build_candidate_pool(all_enriched, period_markers=period_markers or None):
    url = item["attchmntFile"]
    basename = url.rsplit("/", 1)[-1][:40]
    size = item.get("fileSize") or "?"
    meta = item["_metadata_score"]
    try:
        cls = classify_pdf_url(url, session)
        is_fr = cls["is_financial_report"]
        conf = cls["confidence"]
    except Exception as exc:
        is_fr = False
        conf = 0.0
        print(f"  download failed for {basename}: {exc}")
        continue
    bucket = (item.get("event_bucket") or item.get("desc") or "")[:35]
    print(f"{bucket:<35} {basename:<40} {size:>10} {meta:>5.1f} {str(is_fr):>6} {conf:>5.2f}")

print("\n" + "=" * 60)
print("1) Primary check — Financial Results bucket PDFs\n")

for item in recategorized.get("Financial Results", []):
    url = item["attchmntFile"]
    cls = classify_pdf_url(url, session)
    basename = url.rsplit("/", 1)[-1]
    print(f"  {basename}: is_financial_report={cls['is_financial_report']} conf={cls['confidence']} kind={cls['document_kind']}")

print("\n2) Mislink recovery — start from Record Date / AGM row\n")

mislinked = [
    item for item in all_enriched
    if item.get("event_bucket") in ("Record Date", "AGM Intimation")
]
for item in mislinked:
    url = item["attchmntFile"]
    cls = classify_pdf_url(url, session)
    print(f"  {item['event_bucket']}: {url.rsplit('/', 1)[-1]} -> is_fr={cls['is_financial_report']}")

print("\n  Running resolve_financial_report_pdf on full batch...")
resolved = resolve_financial_report_pdf(
    all_enriched,
    period_markers=period_markers or None,
    session=session,
)
if resolved:
    print(f"  Resolved: {resolved['url'].rsplit('/', 1)[-1]}")
    print(f"  From announcement: {resolved['announcement'].get('event_bucket')} / {resolved['announcement'].get('attchmntText', '')[:80]}...")
    print(f"  confidence={resolved['classification']['confidence']} hash={resolved['pdf_hash'][:16]}...")
    saved = save_resolved_financial_pdf(resolved, resolved["pdf_bytes"])
    print(f"  Saved to DB: {saved}")
    _print_processing(saved.get("processing") or {})
else:
    print("  No financial report found.")

print("\n3) Link map — announcements vs canonical report URL\n")
if resolved:
    canonical_url = resolved["url"]
    canonical_hash = resolved["pdf_hash"]
    for item in all_enriched:
        url = item.get("attchmntFile") or ""
        if not url:
            continue
        if url == canonical_url:
            role = "canonical"
        elif url in _PDF_CLASSIFICATION_CACHE and _PDF_CLASSIFICATION_CACHE[url][0] == canonical_hash:
            role = "same_pdf_different_url"
        else:
            try:
                cls = classify_pdf_url(url, session)
                role = "mislinked_letter" if not cls["is_financial_report"] else "other_financial_pdf"
            except Exception:
                role = "unknown"
        print(f"  [{role:>20}] {item.get('event_bucket', '?'):<25} {url.rsplit('/', 1)[-1]}")

# Generic sanity checks (no symbol-specific URLs)
if resolved:
    assert resolved["classification"]["is_financial_report"], "resolved PDF must pass content check"
    assert resolved["classification"]["signals"].get("has_core_financial"), "resolved PDF must contain core financial sections"
    assert "monitoring agency" not in resolved["url"].lower(), "must not pick monitoring agency report"
    print("\nSanity checks passed.")
else:
    print("\nNo financial report resolved for this symbol/date window.")


Symbol: OMKARCHEM  |  Period markers inferred: ['31 march 2026', 'march 31 2026']

PDF classification summary:

event_bucket                        basename                                       size  meta  is_fr  conf
---------------------------------------------------------------------------------------------------------
Financial Results                   OMKARCHEM_12062026204832_OMKARCHEM-Board    6.10 MB  14.0   True  1.00
Financial Results                   OMKARCHEM_14062026165448_Intimation_News    7.00 MB   6.5  False  0.13
Newspaper Publication               OMKARCHEM_02062026214849_Newspaper_Publi    5.08 MB   3.0  False  0.49

1) Primary check — Financial Results bucket PDFs

  OMKARCHEM_14062026165448_Intimation_Newspaper_Publication.pdf: is_financial_report=False conf=0.13 kind=OTHER
  OMKARCHEM_12062026204832_OMKARCHEM-Board_meeting_outcome_12-06-2026.pdf: is_financial_report=True conf=1.0 kind=FINANCIAL_RESULT

2) Mislink recovery — start from Record Date / AGM row


  

process_document failed for 905369d33b2444ff1b8722916c577769caa1297e80d3205c31a0dfdbabfc21bb
Traceback (most recent call last):
  File "/Users/prairit/capital-nerve/event_type_classification/../v3/pipeline/runner.py", line 64, in process_document
    extraction = run_extraction(
                 ^^^^^^^^^^^^^^^
  File "/Users/prairit/capital-nerve/event_type_classification/../v3/pipeline/extract.py", line 257, in run_extraction
    raise RuntimeError("Could not detect reporting period from PDF or title")
RuntimeError: Could not detect reporting period from PDF or title


  Saved to DB: {'company_id': 'ff02713d93bb0b198c1d3a32dfbbab5e5e96e066184fb3b262c494ec05131188', 'document_id': '905369d33b2444ff1b8722916c577769caa1297e80d3205c31a0dfdbabfc21bb', 'event_id': 'f6cc7c752adefaf717e9483b60fedf89e3a78731b40a513c259a0b4bee1fea0d', 'storage_path': '/Users/prairit/capital-nerve/v3/data/documents/905369d33b2444ff1b8722916c577769caa1297e80d3205c31a0dfdbabfc21bb.pdf', 'already_existed': True, 'processing': {'success': False, 'document_id': '905369d33b2444ff1b8722916c577769caa1297e80d3205c31a0dfdbabfc21bb', 'event_id': 'f6cc7c752adefaf717e9483b60fedf89e3a78731b40a513c259a0b4bee1fea0d', 'status': 'failed', 'error': 'Could not detect reporting period from PDF or title'}}
  Processing failed: Could not detect reporting period from PDF or title

3) Link map — announcements vs canonical report URL

  [    mislinked_letter] Financial Results         OMKARCHEM_14062026165448_Intimation_Newspaper_Publication.pdf
  [           canonical] Financial Results         OMKARCH

In [7]:
# Financial Results bucket: verify PDF content and recover if mislinked

import importlib
import sys
import json
from pathlib import Path

sys.path.insert(0, str(Path("..") / "v3"))
import store

importlib.reload(store)
from store import save_resolved_financial_pdf


def _print_processing(proc: dict) -> None:
    if proc.get("success"):
        print(
            f"  Processing: {proc.get('extracted_values', 0)} extracted_values, "
            f"{proc.get('metric_values', 0)} metric_values, {proc.get('signals', 0)} signals"
        )
    elif proc:
        print(f"  Processing failed: {proc.get('error')}")
    else:
        print("  Processing: not run (restart kernel if you just updated v3 code)")


def _sanitize_for_json(obj):
    if isinstance(obj, dict):
        return {k: _sanitize_for_json(v) for k, v in obj.items() if k != "pdf_bytes"}
    if isinstance(obj, list):
        return [_sanitize_for_json(v) for v in obj]
    return obj


all_enriched_fr = [item for items in recategorized.values() for item in items]
period_markers_fr = infer_period_markers(all_enriched_fr)

financial_results_verified: list[dict] = []

for item in recategorized.get("Financial Results", []):
    url = item.get("attchmntFile")
    entry = dict(item)
    if not url:
        entry["pdf_classification"] = None
        entry["resolved_financial_report"] = None
        financial_results_verified.append(entry)
        continue

    cls = classify_pdf_url(url, session)
    entry["pdf_classification"] = cls

    if cls["is_financial_report"]:
        digest, _, pdf_bytes = _PDF_CLASSIFICATION_CACHE[url]
        entry["resolved_financial_report"] = {
            "url": url,
            "announcement": item,
            "classification": cls,
            "pdf_hash": digest,
            "pdf_bytes": pdf_bytes,
            "metadata_score": metadata_score(item),
            "recovery_needed": False,
        }
        saved = save_resolved_financial_pdf(entry["resolved_financial_report"], pdf_bytes)
        proc = saved.get("processing") or {}
        entry["processing"] = proc
        _print_processing(proc)
    else:
        resolved = resolve_financial_report_pdf(
            all_enriched_fr,
            period_markers=period_markers_fr or None,
            session=session,
            tried_urls={url},
        )
        if resolved:
            resolved["recovery_needed"] = True
            resolved["rejected_url"] = url
            saved = save_resolved_financial_pdf(resolved, resolved["pdf_bytes"])
            proc = saved.get("processing") or {}
            resolved["processing"] = proc
            _print_processing(proc)
        entry["resolved_financial_report"] = resolved

    financial_results_verified.append(entry)

print(f"Found {len(financial_results_verified)} Financial Results announcements:\n")
if not financial_results_verified:
    batch_resolved = resolve_financial_report_pdf(
        all_enriched_fr,
        period_markers=period_markers_fr or None,
        session=session,
    )
    if batch_resolved:
        print("(No Financial Results bucket — batch-level canonical report)\n")
        saved = save_resolved_financial_pdf(batch_resolved, batch_resolved["pdf_bytes"])
        proc = saved.get("processing") or {}
        if proc.get("success"):
            print(
                f"  Processing: {proc.get('values', 0)} values, "
                f"{proc.get('metric_values', 0)} metrics, {proc.get('signals', 0)} signals"
            )

for fr in financial_results_verified:
    cls = fr.get("pdf_classification") or {}
    resolved = fr.get("resolved_financial_report")
    basename = (fr.get("attchmntFile") or "").rsplit("/", 1)[-1]
    print(f"--- {basename} ---")
    print(f"  attchmntText: {fr.get('attchmntText', '')[:100]}...")
    print(f"  pdf is_financial_report: {cls.get('is_financial_report')} (conf={cls.get('confidence')})")
    if resolved:
        rec_url = resolved.get("url", "")
        print(f"  canonical URL: {rec_url.rsplit('/', 1)[-1] if rec_url else 'none'}")
        if resolved.get("recovery_needed"):
            print(f"  ** recovered from rejected: {resolved.get('rejected_url', '').rsplit('/', 1)[-1]}")
    proc = fr.get("processing") or (resolved or {}).get("processing") or {}
    if proc:
        _print_processing(proc)
    print()

print(json.dumps(_sanitize_for_json(financial_results_verified), indent=2))


process_document failed for 905369d33b2444ff1b8722916c577769caa1297e80d3205c31a0dfdbabfc21bb
Traceback (most recent call last):
  File "/Users/prairit/capital-nerve/event_type_classification/../v3/pipeline/runner.py", line 64, in process_document
    extraction = run_extraction(
                 ^^^^^^^^^^^^^^^
  File "/Users/prairit/capital-nerve/event_type_classification/../v3/pipeline/extract.py", line 257, in run_extraction
    raise RuntimeError("Could not detect reporting period from PDF or title")
RuntimeError: Could not detect reporting period from PDF or title
process_document failed for 905369d33b2444ff1b8722916c577769caa1297e80d3205c31a0dfdbabfc21bb
Traceback (most recent call last):
  File "/Users/prairit/capital-nerve/event_type_classification/../v3/pipeline/runner.py", line 64, in process_document
    extraction = run_extraction(
                 ^^^^^^^^^^^^^^^
  File "/Users/prairit/capital-nerve/event_type_classification/../v3/pipeline/extract.py", line 257, in run_ext

Found 2 Financial Results announcements:

--- OMKARCHEM_14062026165448_Intimation_Newspaper_Publication.pdf ---
  attchmntText: Omkar Speciality Chemicals Limited has informed the Exchange about Copy of Newspaper Publication for...
  pdf is_financial_report: False (conf=0.13)
  canonical URL: OMKARCHEM_12062026204832_OMKARCHEM-Board_meeting_outcome_12-06-2026.pdf
  ** recovered from rejected: OMKARCHEM_14062026165448_Intimation_Newspaper_Publication.pdf

--- OMKARCHEM_12062026204832_OMKARCHEM-Board_meeting_outcome_12-06-2026.pdf ---
  attchmntText: Omkar Speciality Chemicals Limited has submitted to the Exchange, the financial results for the peri...
  pdf is_financial_report: True (conf=1.0)
  canonical URL: OMKARCHEM_12062026204832_OMKARCHEM-Board_meeting_outcome_12-06-2026.pdf

[
  {
    "symbol": "OMKARCHEM",
    "desc": "Copy of Newspaper Publication",
    "dt": "14062026165506",
    "attchmntFile": "https://nsearchives.nseindia.com/corporate/OMKARCHEM_14062026165448_Intimation_Ne